# Laboratory Exercise: Constructing a Goal-Based Agent using an LLM
**Course:** Undergraduate Artificial Intelligence  
**Focus:** Goal-Based Agents, State Space Search, Problem Formulation, and Prompt Engineering  

---

## 1. Learning Objectives
By completing this laboratory, you will be able to:
1. Explain the operation of a **goal-based intelligent agent** and contrast it with a simple reflex agent;
2. Use a Large Language Model (LLM) as a software engineering assistant;
3. Formulate, generate, execute, and test Python code for grid-world navigation;
4. Improve software quality and algorithm correctness through iterative prompt engineering;
5. Critically evaluate the strengths, assumptions, and limitations of LLM-assisted software development.

---

## 2. Background
In artificial intelligence (Russell & Norvig, AIMA Ch. 2), agents are characterized by their decision-making architecture:
* **Simple Reflex Agents:** Select actions based solely on the *current percept*, ignoring the history of percepts and future consequences. They rely on condition-action rules (`if condition then action`).
* **Model-Based Reflex Agents:** Maintain an internal state to track aspects of the unobserved world.
* **Goal-Based Agents:** Combine knowledge of the current state with an explicit **goal** (a description of desirable situations). To decide on an action, the agent combines goal information with models of action outcomes (search and planning).

In this exercise, the agent operates in a warehouse grid environment containing impassable obstacles (shelving units). Its objective is to find a collision-free path from a loading bay (Start `S`) to a dispatch area (Goal `G`).


## 3. The Warehouse Navigation Problem
The warehouse is represented by a discrete 2D grid:
* `S` denotes the starting position (Loading Bay).
* `G` denotes the goal position (Dispatch Area).
* `#` denotes an obstacle (Shelving unit, impassable).
* `.` denotes free space (Passable floor).

Allowed movements:
* **Up** $(-1, 0)$
* **Down** $(+1, 0)$
* **Left** $(0, -1)$
* **Right** $(0, +1)$

Each movement transitions the agent by one grid cell with unit cost $c = 1$.

```text
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
```


---
## Task 1: Understanding the Problem
Before writing any software, we analyze the theoretical structure of the problem.

### 1. What is the environment?
* **Type:** Static, discrete, fully observable, deterministic, single-agent grid world.
* **Dimensions:** 6 rows by 21 columns ($6 \times 21 = 126$ grid cells).
* **Boundaries & Obstacles:** Impassable outer walls and internal shelving units marked by `'#'`. Free traversable tiles are marked by `'.'`.

### 2. What is the goal of the agent?
* The agent's goal is to transition from its initial coordinate $S = (1, 1)$ to the target destination coordinate $G = (1, 19)$ without colliding with any obstacle cell `'#'` or stepping out of the grid boundaries.

### 3. What actions are available to the agent?
* A set of four orthogonal cardinal movements:
  $$\mathcal{A} = \{\text{Up}, \text{Down}, \text{Left}, \text{Right}\}$$
* Coordinate transitions:
  * $\text{Up}: (r, c) \mapsto (r - 1, c)$
  * $\text{Down}: (r, c) \mapsto (r + 1, c)$
  * $\text{Left}: (r, c) \mapsto (r, c - 1)$
  * $\text{Right}: (r, c) \mapsto (r, c + 1)$
* An action is valid if and only if the destination cell $(r', c')$ satisfies:
  $$0 \le r' < H, \quad 0 \le c' < W, \quad \text{grid}[r'][c'] \neq \text{'\#'}$$

### 4. What information must the agent maintain in order to choose its next action?
To make sound decisions, the agent must maintain:
1. **Current State:** The robot's present grid coordinate $(r, c)$.
2. **Environment Map / Model:** A representation of traversable vs. obstacle cells.
3. **Goal Specification:** The target location $(r_G, c_G)$.
4. **Search State / History:**
   * A **Frontier** (open set) of candidate states discovered but not yet expanded.
   * An **Explored / Visited Set** (closed set) to prevent cycles and redundant expansions.
   * **Parent / Predecessor Pointers** to reconstruct the sequence of actions once the goal is reached.

### 5. Why is this an example of a goal-based agent rather than a simple reflex agent?
A simple reflex agent acts strictly on immediate sensory percepts (e.g., *"if the cell directly ahead is free, move forward"*). In an obstacle-rich maze with dead-ends, U-shaped walls, and corridors, a reflex agent inevitably becomes trapped in local loops or oscillates endlessly because it cannot evaluate the long-term consequences of actions.  
In contrast, a **goal-based agent** explicitly evaluates sequences of future actions against a desired final state $G$. It performs search over simulated outcomes to formulate an entire plan before acting, guaranteeing navigation around obstacles to reach the destination.

---
### Think About It: Scaling to a Warehouse Twice as Large
* **Would the same search strategy still be appropriate?**
  * If using **Breadth-First Search (BFS)**, the search complexity is $O(b^d)$, where $b$ is the effective branching factor and $d$ is the path depth. Doubling the grid dimensions quadruples the area ($4 \times$) and roughly doubles the search depth $d$. For BFS, the memory requirement explodes exponentially because it expands nodes uniformly in all directions.
* **What additional difficulties arise?**
  1. **Combinatorial State Space Explosion:** High memory consumption for storing the frontier and visited sets.
  2. **Need for Informed Search:** Uninformed BFS becomes impractical; heuristic-guided search like **$A^*$ Search** (using Manhattan distance) is required to focus exploration directly toward the target.
  3. **Hierarchical / Multi-Resolution Planning:** Very large industrial warehouses require hierarchical planning (e.g., waypoint navigation between aisle intersections rather than raw cell-by-cell BFS).


---
## Task 2: Designing the Agent
Before prompting an LLM for code, we specify the agent's architectural decomposition.

### Architectural Components:
1. **Environment:** 2D grid containing obstacles (`#`), traversable floor (`.`), start (`S`), and goal (`G`).
2. **Percepts / State:** The coordinates $(r, c)$ of the agent in the grid.
3. **Transition Model:** $T(s, a) \to s'$, determining the resulting coordinate after taking movement $a$.
4. **Goal Test:** A boolean predicate $\text{IsGoal}(s) \equiv (s == (r_G, c_G))$.
5. **Decision-Making Component (Search Engine):**
   * Employs Breadth-First Search (BFS) to guarantee the shortest path under uniform action costs ($c=1$).
   * Uses a FIFO queue for the frontier, a set for visited states, and a dictionary of parent pointers for path reconstruction.

### Architecture Block Diagram:
```text
+---------------------------------------------------------------------------------+
|                               GOAL-BASED AGENT                                  |
|                                                                                 |
|   +---------------+      +----------------------+      +--------------------+   |
|   |  Environment  | ---> |   State Estimator    | ---> |   Current State    |   |
|   |   Percepts    |      | (Extract Coordinates)|      |      s = (r, c)    |   |
|   +---------------+      +----------------------+      +--------------------+   |
|                                                                   |             |
|                                                                   v             |
|   +---------------+      +----------------------+      +--------------------+   |
|   |  Goal State   | ---> |  Search Algorithm    | <--- |  Transition Model  |   |
|   |  G = (rG, cG) |      | (BFS Path Planner)   |      |  T(s, a) & Actions |   |
|   +---------------+      +----------------------+      +--------------------+   |
|                                     |                                           |
|                                     v                                           |
|                          +----------------------+                               |
|                          | Planned Action Plan  |                               |
|                          |  [a1, a2, ..., an]   |                               |
|                          +----------------------+                               |
|                                     |                                           |
|                                     v                                           |
|                          +----------------------+                               |
|                          |  Actuator Execution  | ---> Environment Action       |
|                          +----------------------+                               |
+---------------------------------------------------------------------------------+
```


---
## Task 3: Prompt Engineering & Implementation
We formulated the following clear, structured prompt based on our formal design specification:

### Specification Prompt Provided to the LLM:
```text
Write a well-documented Python program implementing a goal-based agent for the warehouse navigation problem.
The environment is defined by the following ASCII map:
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################

The program should:
1. Represent the warehouse as a two-dimensional grid;
2. Automatically locate start 'S' and goal 'G';
3. Determine a collision-free path from S to G using an optimal search algorithm (BFS);
4. Avoid all obstacle cells '#';
5. Reconstruct and print the step-by-step path coordinates and action sequence;
6. Print the visual warehouse map overlaying the path with '*' markers;
7. Explain the search algorithm chosen and why it is appropriate.
```


In [1]:
"""
Warehouse Goal-Based Navigation Agent
Implementation using Breadth-First Search (BFS)
"""
from collections import deque
from typing import List, Tuple, Optional, Dict

class WarehouseEnvironment:
    """Represents the 2D grid warehouse environment."""
    def __init__(self, map_str: str):
        self.grid = [list(line.strip()) for line in map_str.strip().splitlines() if line.strip()]
        self.height = len(self.grid)
        self.width = len(self.grid[0])
        self.start = self._find_symbol('S')
        self.goal = self._find_symbol('G')

        if not self.start or not self.goal:
            raise ValueError("Map must contain both 'S' (start) and 'G' (goal) symbols.")

    def _find_symbol(self, symbol: str) -> Optional[Tuple[int, int]]:
        for r in range(self.height):
            for c in range(self.width):
                if self.grid[r][c] == symbol:
                    return (r, c)
        return None

    def is_valid_move(self, r: int, c: int) -> bool:
        """Checks if cell is within bounds and not an obstacle."""
        return 0 <= r < self.height and 0 <= c < self.width and self.grid[r][c] != '#'

    def get_neighbors(self, state: Tuple[int, int]) -> List[Tuple[Tuple[int, int], str]]:
        """Returns valid adjacent (state, action) pairs."""
        r, c = state
        moves = [
            ((-1, 0), "Up"),
            ((1, 0), "Down"),
            ((0, -1), "Left"),
            ((0, 1), "Right")
        ]
        valid_transitions = []
        for (dr, dc), action_name in moves:
            nr, nc = r + dr, c + dc
            if self.is_valid_move(nr, nc):
                valid_transitions.append(((nr, nc), action_name))
        return valid_transitions

    def render_path(self, path: List[Tuple[int, int]]) -> str:
        """Renders the grid with the solution path marked with '*'."""
        display_grid = [row.copy() for row in self.grid]
        path_set = set(path)
        for (r, c) in path:
            if (r, c) != self.start and (r, c) != self.goal:
                display_grid[r][c] = '*'
        return "\n".join("".join(row) for row in display_grid)


class GoalBasedAgent:
    """A goal-based agent implementing Breadth-First Search (BFS)."""
    def __init__(self, env: WarehouseEnvironment):
        self.env = env
        self.states_expanded = 0

    def plan_path(self) -> Optional[Tuple[List[Tuple[int, int]], List[str]]]:
        """
        Plans a shortest path from start to goal using BFS.
        Returns:
            (path_coordinates, action_sequence) if path found, else None.
        """
        start = self.env.start
        goal = self.env.goal

        # FIFO frontier storing current state
        frontier = deque([start])
        # Visited set to avoid cycles
        visited = {start}
        # Parent mapping: child -> (parent, action)
        parent_map: Dict[Tuple[int, int], Tuple[Optional[Tuple[int, int]], Optional[str]]] = {
            start: (None, None)
        }

        self.states_expanded = 0

        while frontier:
            current = frontier.popleft()
            self.states_expanded += 1

            # Goal Test
            if current == goal:
                return self._reconstruct_path(parent_map, goal)

            # Expand successors
            for next_state, action in self.env.get_neighbors(current):
                if next_state not in visited:
                    visited.add(next_state)
                    parent_map[next_state] = (current, action)
                    frontier.append(next_state)

        return None  # No path exists

    def _reconstruct_path(self, parent_map, goal):
        path = []
        actions = []
        curr = goal
        while curr is not None:
            path.append(curr)
            parent, act = parent_map[curr]
            if act is not None:
                actions.append(act)
            curr = parent
        path.reverse()
        actions.reverse()
        return path, actions


# --- Execution and Verification ---
WAREHOUSE_MAP = """
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
"""

env = WarehouseEnvironment(WAREHOUSE_MAP)
agent = GoalBasedAgent(env)

print(f"Warehouse Dimensions: {env.height} rows x {env.width} columns")
print(f"Start Position (S): {env.start}")
print(f"Goal Position  (G): {env.goal}")
print("-" * 55)

result = agent.plan_path()

if result:
    path, actions = result
    print("SUCCESS: Path to goal successfully found!")
    print(f"Path Length (Number of Steps): {len(actions)}")
    print(f"States Expanded: {agent.states_expanded}")
    print("-" * 55)
    print("Action Sequence:")
    print(" -> ".join(actions))
    print("-" * 55)
    print("Grid with Solution Trajectory ('*' marks path):")
    print(env.render_path(path))
else:
    print("FAILURE: No collision-free path exists to the goal.")


Warehouse Dimensions: 7 rows x 21 columns
Start Position (S): (1, 1)
Goal Position  (G): (1, 19)
-------------------------------------------------------
SUCCESS: Path to goal successfully found!
Path Length (Number of Steps): 20
States Expanded: 59
-------------------------------------------------------
Action Sequence:
Right -> Right -> Right -> Down -> Right -> Right -> Right -> Up -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right
-------------------------------------------------------
Grid with Solution Trajectory ('*' marks path):
#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


---
## Task 3 Review & Reflection Questions

### 1. Did the LLM generate a working program on the first attempt?
**Yes.** Because our prompt was architecturally constrained (specifying grid coordinates, boundary validation, FIFO frontier, cycle avoidance via a visited set, and explicit parent-pointer path reconstruction), the resulting program executed cleanly without syntax errors, boundary faults, or index out-of-bounds exceptions.

### 2. If not, how can you improve your prompt?
Common failure modes when prompting for search agents include:
* Using greedy coordinate steps without maintaining an explored set (causing infinite oscillation in corners).
* Transposing row/column indices (e.g., treating `(x, y)` as `(col, row)` instead of `(row, col)`).
* Hardcoding grid dimensions or failing to check boundaries.

To eliminate these issues, prompts must specify:
* Explicit coordinate conventions: `(row, column)` where `0 <= row < height` and `0 <= col < width`.
* Explicit algorithm selection (e.g., *"use Breadth-First Search with collections.deque and an explored set"*).
* Exact input/output contracts (e.g., *"return (path_coordinates, action_list)"*).

### 3. What search algorithm did the LLM choose?
The implementation uses **Breadth-First Search (BFS)** across the 2D grid graph.

### 4. Why do you think the LLM selected this algorithm?
1. **Guaranteed Optimality:** In an unweighted grid world where every step has an identical cost ($c = 1$), BFS is mathematically guaranteed to find the shortest possible path (minimal number of steps).
2. **Completeness:** On finite graphs, BFS is complete—it will always find a path if one exists, or conclusively terminate if no path exists.
3. **Simplicity and Robustness:** BFS requires only standard Python libraries (`collections.deque`, `set`, `dict`) without needing a priority queue or heuristic design.

---

## 4. Key Takeaways
1. **Reflex vs. Goal-Based:** Navigation in environments with concave obstacles and dead-ends is impossible for simple reflex agents; goal-based formulation and search over state spaces is essential.
2. **AI Science vs. AI Engineering:** 
   * *AI Science* defines state spaces $(S, A, T, s_0, G)$ and formal optimality guarantees.
   * *AI Engineering* designs modular data structures, tests corner cases, and uses LLMs to accelerate boilerplate generation while independently verifying code behavior against formal requirements.
